# Lesson 5: Train the bigram model

We now let the model learn from the next-character examples. The model still has the same simple vocabulary-by-vocabulary table; training changes its scores so observed next characters become more likely.

The original tutorial introduces the optimizer and training loop in cells 14–16. Python uses PyTorch autograd and AdamW. The matching [C entry point](../c/src/bigram_training_main.c) computes the gradient and AdamW update explicitly; the [Java entry point](../java/src/main/java/learning/karpathy/gpt1/BigramTrainingLesson.java) does the same with arrays. Use those files to compare the library abstraction with the underlying operations.

## Prepare the corpus and mini-batch sampler

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

# Locate the same corpus from the repo root, gpt-1/, or this Python folder.
data_candidates = (
    Path.cwd() / "data" / "input.txt",
    Path.cwd() / "gpt-1" / "data" / "input.txt",
    Path.cwd().parent / "data" / "input.txt",
)
dataset_path = next((path.resolve() for path in data_candidates if path.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError("Could not find gpt-1/data/input.txt. Open the repo root, gpt-1/, or gpt-1/python/.")

text = dataset_path.read_text(encoding="utf-8")
chars = sorted(set(text))
stoi = {character: token_id for token_id, character in enumerate(chars)}
itos = {token_id: character for token_id, character in enumerate(chars)}
encode = lambda sample: [stoi[character] for character in sample]
decode = lambda token_ids: "".join(itos[token_id] for token_id in token_ids)
data = torch.tensor(encode(text), dtype=torch.long)

split_index = int(0.9 * len(data))
train_data = data[:split_index]
val_data = data[split_index:]
batch_size = 32
block_size = 8

def get_batch(split):
    """Choose random short windows and pair each character with its next character."""
    source = train_data if split == "train" else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x_batch = torch.stack([source[start:start + block_size] for start in starts])
    y_batch = torch.stack([source[start + 1:start + block_size + 1] for start in starts])
    return x_batch, y_batch

print(f"Dataset: {dataset_path}")
print(f"Vocabulary: {len(chars)} characters; train/validation: {len(train_data):,}/{len(val_data):,} tokens")
print(f"Each training batch has shape [{batch_size}, {block_size}] for both inputs and targets.")

## Step 1: Record the starting loss

Before updating any parameters, record loss on two fixed mini-batches. We reuse these same examples later so the before/after comparison is meaningful. The validation batch comes from the final 10% of the text, which the optimizer will not train on.

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # A lookup table: one row per current character, one score per possible next character.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx is (B, T); one lookup gives logits with shape (B, T, V).
        logits = self.token_embedding_table(idx)
        loss = None
        if targets is not None:
            batch, time, vocab = logits.shape
            # Flatten positions so cross-entropy sees (positions, classes) and (positions,).
            loss = F.cross_entropy(
                logits.reshape(batch * time, vocab),
                targets.reshape(batch * time),
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        # Append one sampled next character at a time (autoregressive generation).
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            probabilities = F.softmax(logits[:, -1, :], dim=-1)
            next_id = torch.multinomial(probabilities, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx

torch.manual_seed(1337)
model = BigramLanguageModel(len(chars))

# Fixed examples let us compare the same inputs before and after training.
torch.manual_seed(2025)
train_eval_batch = get_batch("train")
validation_eval_batch = get_batch("validation")

def evaluate(batch):
    inputs, targets = batch
    _, loss = model(inputs, targets)
    return loss.item()

initial_train_loss = evaluate(train_eval_batch)
initial_validation_loss = evaluate(validation_eval_batch)

print(f"Model parameter table: {tuple(model.token_embedding_table.weight.shape)}")
print(f"Fixed evaluation batch shapes: x={tuple(train_eval_batch[0].shape)}, y={tuple(train_eval_batch[1].shape)}")
print(f"Initial loss: train {initial_train_loss:.4f}; validation {initial_validation_loss:.4f}")

## Step 2: Train with cross-entropy, backpropagation, and AdamW

The training loop has four important actions: sample a batch, calculate the loss, clear old gradients, then backpropagate and update the parameters. PyTorch calculates the derivatives during loss.backward() and optimizer.step() applies AdamW.

In C and Java, the model uses the softmax cross-entropy derivative probability minus one_hot(target), accumulates gradients into the selected table rows, then applies AdamW's moving averages and weight decay.

In [ ]:
learning_rate = 1e-3
weight_decay = 0.01
training_steps = 100
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)

torch.manual_seed(2026)
training_history = [(0, initial_train_loss, initial_validation_loss)]
print("step | train loss | validation loss")
print(f"{0:4d} | {initial_train_loss:10.4f} | {initial_validation_loss:15.4f}")

for step in range(1, training_steps + 1):
    x_batch, y_batch = get_batch("train")
    _, loss = model(x_batch, y_batch)

    optimizer.zero_grad(set_to_none=True)  # discard gradients from the previous mini-batch
    loss.backward()                       # autograd computes d(loss)/d(parameter)
    optimizer.step()                      # AdamW updates the embedding table

    if step % 25 == 0:
        train_loss = evaluate(train_eval_batch)
        validation_loss = evaluate(validation_eval_batch)
        training_history.append((step, train_loss, validation_loss))
        print(f"{step:4d} | {train_loss:10.4f} | {validation_loss:15.4f}")

final_train_loss = training_history[-1][1]
assert final_train_loss < initial_train_loss, "Training did not reduce the fixed training-batch loss."

## Step 3: Sample from the trained model

The trained table now assigns higher probability to character pairs seen in the training text. Generation still uses only the immediately preceding character, so it cannot keep track of longer words, names, or plot.

In [ ]:
torch.manual_seed(1337)
start = torch.zeros((1, 1), dtype=torch.long)  # token ID 0 (often a newline)
sampled_ids = model.generate(start, max_new_tokens=200)[0].tolist()
generated_text = decode(sampled_ids)

print(f"Start token: {start.item()} ({decode([start.item()])!r})")
print(generated_text)
assert len(sampled_ids) == 201

## Compare and experiment

Run **Terminal → Run Task → Run tutorial lesson**, then select lesson **5** to compare the C and Java output. Step 3 runs its training prerequisite quietly when selected by itself, so it still generates from a trained model.

Try 10, 100, and 500 training steps. How do the training and validation losses move? What does the gap between them suggest? Then compare loss.backward() and optimizer.step() with the explicit gradient and AdamW code in the C and Java versions.

References: [PyTorch autograd](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html), [AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html), and the [original GPT tutorial notebook](../gpt_dev.ipynb), cells 14–16.